# Hari 2 — Text Preprocessing & Normalisasi
**Project:** Indonesian Toxic Comment Classifier  
**Tujuan hari ini:** Bersihkan teks mentah Twitter agar siap diproses model. Teks kotor = model buruk, sebagus apapun algoritmanya.

---
### Kenapa preprocessing penting untuk teks Twitter?
Tweet Indonesia punya banyak noise:
- `@USER`, `RT USER:` → artefak Twitter, bukan konten
- `http://...` → URL tidak bermakna untuk klasifikasi
- `"gw"`, `"lu"`, `"yg"` → slang yang perlu dinormalisasi
- Huruf kapital → `"BRENGSEK"` dan `"brengsek"` harus dianggap sama

Urutan preprocessing yang akan kita bangun hari ini:
```
Teks mentah
    → lowercase
    → hapus URL
    → hapus mention & RT
    → hapus karakter non-alfabet
    → normalisasi slang (kamus kamusalay)
    → hapus whitespace berlebih
→ Teks bersih
```

## 0. Import Libraries

In [1]:
import pandas as pd
import re        # regex — untuk pattern matching di teks
import string    # kumpulan karakter seperti tanda baca

---
## 1. Load Ulang Dataset

**Konteks:** Kita load ulang dari file asli agar notebook ini bisa dijalankan mandiri tanpa bergantung pada notebook EDA.
Ini kebiasaan baik — setiap notebook harus self-contained.

**Instruksi:**
- Load `../data/data.csv` dengan encoding `latin-1`, simpan ke `df`
- Drop duplikat langsung
- Print shape setelah drop

In [2]:
# TODO: load dataset & drop duplikat
df = pd.read_csv("../data/data.csv", encoding='latin-1')
df = df.drop_duplicates()

print("Shape:", df.shape)

Shape: (13044, 13)


---
## 2. Load Kamus Slang (kamusalay)

**Konteks:** File `new_kamusalay.csv` tidak punya header. Isinya dua kolom:
- Kolom 0: kata slang/typo (contoh: `"gw"`, `"yg"`, `"bgt"`)
- Kolom 1: bentuk normalnya (contoh: `"saya"`, `"yang"`, `"banget"`)

Kita akan ubah ini menjadi Python dictionary agar lookup-nya cepat: `{"gw": "saya", "yg": "yang", ...}`

**Instruksi:**
- Load `../data/new_kamusalay.csv` dengan `header=None` dan encoding `latin-1`
- Simpan ke variabel `kamus_df`
- Tampilkan 5 baris pertama untuk verifikasi
- Ubah menjadi dictionary menggunakan `dict(zip(kolom_0, kolom_1))`
- Simpan ke variabel `slang_dict`
- Print berapa banyak entri di dictionary

In [3]:
# TODO: load kamus slang
kamus_df = pd.read_csv("../data/new_kamusalay.csv",header=None, encoding='latin-1')

# TODO: tampilkan 5 baris pertama
print(kamus_df.head(5))

# TODO: ubah ke dictionary
# Hint: kamus_df[0] = kolom slang, kamus_df[1] = kolom normal
slang_dict = dict(zip(kamus_df[0] ,kamus_df[1]))

print(f"Jumlah entri kamus: {len(slang_dict)}")
# TODO: print 3 contoh isi dictionary untuk verifikasi
# Hint: dict tidak bisa di-index, tapi bisa di-convert ke list dulu
# contoh: list(slang_dict.items())[:3]

contoh = list(slang_dict.items())[:3]
print(contoh)

                     0                         1
0  anakjakartaasikasik  anak jakarta asyik asyik
1         pakcikdahtua         pak cik sudah tua
2       pakcikmudalagi         pak cik muda lagi
3          t3tapjokowi              tetap jokowi
4                   3x                 tiga kali
Jumlah entri kamus: 15167
[('anakjakartaasikasik', 'anak jakarta asyik asyik'), ('pakcikdahtua', 'pak cik sudah tua'), ('pakcikmudalagi', 'pak cik muda lagi')]


---
## 3. Bangun Fungsi Preprocessing Satu per Satu

**Konteks:** Kita akan bangun setiap langkah sebagai fungsi terpisah, lalu gabungkan di akhir.
Ini prinsip **modular programming** — lebih mudah di-debug dan di-test.

Kita akan banyak pakai `re` (regex) di sini. Pola dasar yang perlu kamu tahu:
| Pattern | Artinya |
|---------|--------|
| `r'http\S+'` | URL yang diawali `http` |
| `r'@\w+'` | mention Twitter seperti `@namauser` |
| `r'\bRT\b'` | kata `RT` yang berdiri sendiri |
| `r'[^a-z ]'` | karakter selain huruf a-z dan spasi |

### 3a. Lowercase

**Instruksi:**
- Buat fungsi `lowercase(text)` yang mengembalikan teks dalam huruf kecil semua
- Gunakan Python string method `.lower()`
- Test dengan contoh string di bawah

In [4]:
def lowercase(text):
    text = text.lower()
    return text

# Test
print(lowercase("DASAR BRENGSEK Lu!"))

dasar brengsek lu!


### 3b. Hapus URL

**Instruksi:**
- Buat fungsi `remove_url(text)` menggunakan `re.sub(pattern, replacement, text)`
- Pattern untuk URL: `r'http\S+'`
- Ganti dengan string kosong `''`
- Test dengan contoh di bawah

In [5]:
def remove_url(text):
    text = re.sub(r'http\S+', '', text)
    return text

# Test
print(remove_url("cek ini https://t.co/abc123 bagus banget"))

cek ini  bagus banget


### 3c. Hapus Mention & RT

**Instruksi:**
- Buat fungsi `remove_mention_rt(text)`
- Hapus mention dengan pattern `r'@\w+'`
- Hapus `RT` dengan pattern `r'\bRT\b'`
- Kamu perlu dua kali `re.sub()` — chain hasilnya
- Test dengan contoh di bawah

In [6]:
def remove_mention_rt(text):
    # TODO: hapus mention
    text = re.sub(r'@\w+', '', text)
    # TODO: hapus RT
    text = re.sub(r'\bRT\b', '', text)
    return text

# Test
print(remove_mention_rt("RT @user123: ini tweet yang di-retweet @someone"))

 : ini tweet yang di-retweet 


### 3d. Hapus Karakter Non-Alfabet

**Konteks:** Angka, tanda baca, dan karakter aneh tidak berkontribusi pada klasifikasi teks.
Kita hanya pertahankan huruf a-z dan spasi.

**Instruksi:**
- Buat fungsi `remove_non_alpha(text)`
- Gunakan pattern `r'[^a-z ]'` — artinya: hapus semua kecuali a-z dan spasi
- **Penting:** fungsi ini harus dipanggil *setelah* lowercase
- Test dengan contoh di bawah

In [16]:
def remove_non_alpha(text):
    # TODO
    text = re.sub(r'[^a-z ]', '', text)
    return text

# Test
print(remove_non_alpha("harga: Rp 50.000 #viral!!! 😡"))

harga p  viral 


### 3e. Normalisasi Slang

**Konteks:** Ini langkah paling unik di dataset Indo. Kita ganti tiap kata yang ada di `slang_dict` dengan bentuk normalnya.

Caranya: pecah teks jadi list kata, cek tiap kata di dictionary, ganti kalau ada, lalu gabung lagi.

**Instruksi:**
- Buat fungsi `normalize_slang(text, slang_dict)`
- Pecah teks dengan `.split()` → list kata
- Untuk tiap kata, gunakan `slang_dict.get(kata, kata)` — artinya: ambil nilai dari dict, kalau tidak ada kembalikan kata aslinya
- Gabung kembali dengan `' '.join(...)`
- Test dengan contoh di bawah

In [8]:
def normalize_slang(text, slang_dict):
    # TODO: pecah jadi list kata
    words = text.split()
    # TODO: ganti tiap kata yang ada di kamus
    # Hint: gunakan list comprehension
    # [slang_dict.get(w, w) for w in words]
    words = [slang_dict.get(w, w) for w in words]
    # TODO: gabung kembali
    text = ' '.join(words)
    return text

# Test
print(normalize_slang("gw lg bgt pengen makan yg enak", slang_dict))

gue lagi banget ingin makan yang enak


### 3f. Hapus Whitespace Berlebih

**Konteks:** Setelah semua penghapusan di atas, teks bisa punya spasi ganda atau leading/trailing space.

**Instruksi:**
- Buat fungsi `remove_extra_space(text)`
- Gunakan `re.sub(r'\s+', ' ', text)` untuk ganti spasi berlebih dengan satu spasi
- Lalu `.strip()` untuk hapus spasi di awal dan akhir
- Test dengan contoh di bawah

In [9]:
def remove_extra_space(text):
    # TODO
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

# Test
print(remove_extra_space("  ini   teks   dengan   spasi   aneh  "))

ini teks dengan spasi aneh


---
## 4. Gabungkan Semua ke Satu Pipeline

**Konteks:** Sekarang kita gabungkan semua fungsi tadi menjadi satu fungsi `preprocess(text)`.
Urutannya penting — lowercase harus pertama, normalisasi slang harus setelah lowercase.

**Instruksi:**
- Buat fungsi `preprocess(text)` yang memanggil semua fungsi secara berurutan
- Urutan yang benar:
  1. lowercase
  2. remove_url
  3. remove_mention_rt
  4. remove_non_alpha
  5. normalize_slang
  6. remove_extra_space
- Test dengan tweet contoh di bawah

In [10]:
def preprocess(text):
    # TODO: panggil semua fungsi secara berurutan
    text = lowercase(text)
    text = remove_url(text)
    text = remove_mention_rt(text)
    text = remove_non_alpha(text)
    text = normalize_slang(text, slang_dict)
    text = remove_extra_space(text)
    return text

# Test dengan tweet nyata dari dataset
contoh = "RT @UserX: Dasar lu brengsek bgt!! https://t.co/abc #toxic 😡"
print("Sebelum:", contoh)
print("Sesudah:", preprocess(contoh))

Sebelum: RT @UserX: Dasar lu brengsek bgt!! https://t.co/abc #toxic 😡
Sesudah: rt dasar kamu berengsek banget toxic


---
## 5. Terapkan ke Seluruh Dataset

**Konteks:** Sekarang kita terapkan fungsi `preprocess()` ke seluruh kolom `Tweet`.
Kita simpan hasilnya ke kolom baru `clean_tweet` agar kolom asli tetap terjaga.

**Instruksi:**
- Gunakan `.apply(preprocess)` pada kolom `Tweet`
- Simpan ke kolom baru `df['clean_tweet']`
- Ini akan memakan beberapa detik karena 13.000+ baris

In [11]:
print("Memproses...")

# TODO: terapkan preprocess ke seluruh kolom Tweet
df['clean_tweet'] = df['Tweet'].apply(preprocess)

print("Selesai!")
print(f"Kolom sekarang: {df.columns.tolist()}")

Memproses...
Selesai!
Kolom sekarang: ['Tweet', 'HS', 'Abusive', 'HS_Individual', 'HS_Group', 'HS_Religion', 'HS_Race', 'HS_Physical', 'HS_Gender', 'HS_Other', 'HS_Weak', 'HS_Moderate', 'HS_Strong', 'clean_tweet']


---
## 6. Verifikasi Hasil Preprocessing

**Konteks:** Selalu verifikasi hasilnya secara manual. Preprocessing yang salah lebih berbahaya dari tidak preprocessing sama sekali — karena errornya tidak langsung kelihatan.

**Instruksi:**
- Tampilkan 10 baris dengan kolom `Tweet` dan `clean_tweet` berdampingan menggunakan `.sample(10)`
- Perhatikan: apakah slang sudah terganti? Apakah URL dan mention sudah hilang?

In [12]:
# TODO: tampilkan perbandingan sebelum dan sesudah
pd.set_option('display.max_colwidth', 80)  # biar kolom tidak terpotong
df[['Tweet', 'clean_tweet']].sample(10)



,Tweet,clean_tweet
11390,USER WISHNYA KE LU SEMUA TAI.',pengguna harapannya ke kamu semua tai
7586,RT USER: Saat semua yg ditakutkan dari Prabowo sudah dilakukan dengan sukses...,rt pengguna saat semua yang ditakutkan dari prabowo sudah dilakukan dengan s...
2353,"1 oktober hari kesaktian pancasila, tapi pancasila sekarang diversikan ke ar...",oktober hari kesaktian pancasila tapi pancasila sekarang diversikan ke arah ...
12415,USER USER USER cacat bgt ican goblok',pengguna pengguna pengguna cacat banget ican goblok
7717,Prabowo makin panik. \nPrabowo makin membabi buta. \nSerang siapa saja. \nEl...,prabowo makin panik prabowo makin membabi buta erang siapa saja elite politi...
501,USER USER USER USER Itu dia mata picek... Matanya cuma lihat sebelah...',pengguna pengguna pengguna pengguna itu dia mata picek matanya cuma lihat se...
9690,Pengamat Unpad: Kritik Prabowo kepada Jokowi Bisa Jadi Blunder USER USER URL,pengamat universitas negeri padjadjaran kritik prabowo kepada jokowi bisa ja...
4365,"Tolak dan lawan Politik uang, Politisasi sara untuk Pilkada yang Berintegrit...",tolak dan lawan politik uang politisasi sara untuk pilihan kepala daerah yan...
7388,"Si cewe itu bilang ""gua baru mau keluar dari badan ini kalo rumah gua udah d...",sih cewek itu bilang gue baru mau keluar dari badan ini kalau rumah gue suda...
4341,USER Kenapa? Terguncang ente mendengarnya pak?? Makanya para ulama zaman dah...,pengguna kenapa terguncang kamu mendengarnya pak makanya para ulama zaman da...


---
## 7. Cek Tweet Kosong Setelah Preprocessing

**Konteks:** Ada kemungkinan tweet yang setelah dibersihkan menjadi string kosong `''` — misalnya tweet yang isinya hanya URL atau mention saja. Tweet kosong harus di-drop karena tidak ada informasi yang bisa dipelajari model.

**Instruksi:**
- Hitung berapa tweet yang `clean_tweet`-nya kosong (`== ''`) atau hanya spasi (`.str.strip() == ''`)
- Drop baris-baris tersebut
- Print jumlah sebelum dan sesudah

In [13]:
print("Sebelum:", len(df))

# TODO: hitung tweet kosong
tweet_kosong = ((df['clean_tweet'].str.strip() == '').sum())
print("Tweet kosong:", tweet_kosong)

# TODO: drop tweet kosong dan simpan ke df
df = df[df['clean_tweet'].str.strip() != '']

print("Sesudah:", len(df))

Sebelum: 13044
Tweet kosong: 0
Sesudah: 13044


---
## 8. Simpan Dataset Bersih

**Konteks:** Kita simpan hasilnya ke file baru. Hari 3 (modeling) akan load dari file ini — bukan dari `data.csv` asli.
Kita hanya perlu dua kolom: `clean_tweet` dan `HS`.

**Instruksi:**
- Buat DataFrame baru `df_clean` yang hanya berisi kolom `clean_tweet` dan `HS`
- Simpan ke `../data/data_clean.csv` menggunakan `.to_csv()` dengan `index=False`
- Load ulang dan tampilkan `.head(3)` untuk konfirmasi file tersimpan dengan benar

In [14]:
# TODO: buat df_clean dengan hanya kolom yang dibutuhkan
df_clean = df[['clean_tweet', 'HS']]
print(df_clean.head(3))

# TODO: simpan ke CSV
df_clean.to_csv('../data/data_clean.csv', index=False)

# Verifikasi — load ulang
verify = pd.read_csv('../data/data_clean.csv')
print("File tersimpan. Shape:", verify.shape)
print(verify.head(3))

                                                                       clean_tweet  \
0  di saat semua cowok berusaha melacak perhatian gue kamu lantas remehkan perh...   
1  rt pengguna pengguna siapa yang telat memberi tau eluedan sarap gue bergaul ...   
2  kadang aku berpikir kenapa aku tetap percaya pada tuhan padahal aku selalu j...   

   HS  
0   1  
1   0  
2   0  


File tersimpan. Shape: (13044, 2)
                                                                       clean_tweet  \
0  di saat semua cowok berusaha melacak perhatian gue kamu lantas remehkan perh...   
1  rt pengguna pengguna siapa yang telat memberi tau eluedan sarap gue bergaul ...   
2  kadang aku berpikir kenapa aku tetap percaya pada tuhan padahal aku selalu j...   

   HS  
0   1  
1   0  
2   0  


---
## 9. Ringkasan Temuan

Jawab dengan kata-katamu sendiri:

1. Kenapa urutan preprocessing itu penting? Apa yang terjadi kalau `normalize_slang` dipanggil sebelum `lowercase`?
2. Dari verifikasi manual di Section 6, apakah ada kasus preprocessing yang hasilnya kurang memuaskan?
3. Berapa tweet yang hilang setelah drop kosong?
4. Menurut kamu, apakah ada langkah preprocessing lain yang mungkin berguna tapi belum kita lakukan?

*(Tulis jawabanmu di sini)*
1. jika normalize_slang dipanggil sebelum lowercase, slang yang sama tapi ditulis kapital/campuran, akan dianggap berbeda dan tidak ternormalisasi
2. ada, kata rt dan pengguna masih banyak di data_clean 
3. tidak ada tweet kosong
4. drop kata yang muncul berulang yang berpotensi menjadi noise

---
## ✅ Checklist Hari 2
- [ ] Kamus slang berhasil diload dan diubah ke dictionary
- [ ] Semua fungsi preprocessing berjalan dan di-test satu per satu
- [ ] Pipeline `preprocess()` diterapkan ke seluruh dataset
- [ ] Hasil preprocessing diverifikasi secara manual
- [ ] Tweet kosong sudah di-drop
- [ ] `data_clean.csv` tersimpan dan bisa dibaca kembali
- [ ] Ringkasan temuan diisi